In [3]:
%pip install torch numpy pandas matplotlib scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [1]:
# ENVIRONMENT
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)

pd.set_option("display.max_columns", None)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print("Environment ready. Torch:", torch.__version__)

Environment ready. Torch: 2.13.0+cpu


# SkiTrickGen: Generative Modeling of Freestyle Aerial Ski Tricks
## A Deep Learning Investigation Using Biomechanically-Grounded Synthetic Kinematics and FIS Degree-of-Difficulty Data

**Project type:** Technical report + reproducible Jupyter notebook
**Main task:** Learn a generative latent representation of freestyle aerial ski maneuvers (flips, twists), conditioned on real FIS trick codes, that can reconstruct known tricks and synthesize novel, physically plausible ones.
**Target representation:** normalized-time kinematic trajectory `[somersault_angle(t), twist_angle(t), tuck_factor(t)]`
**Conditioning signal:** `[n_flips, n_twists, dd_score]` from the real FIS Degree-of-Difficulty table
**Domain:** Freestyle aerial skiing (moguls aerials, big air)

---

## 0. Executive Summary

Freestyle aerial skiing is a discipline in which athletes execute precisely codified combinations of flips and twists during a brief flight phase, judged in part using a Degree of Difficulty (DD) scale published by the International Ski and Snowboard Federation (FIS). Real motion-capture data for this sport is essentially absent from the public domain, which limits coaching tools, animation/game content pipelines, and judging-support software that would benefit from a broader library of trick trajectories than exists on film.

This notebook builds **SkiTrickGen**, a Conditional Variational Autoencoder (CVAE) trained on a biomechanically-grounded *synthetic* kinematic dataset -- generated from a rigid-body flight-dynamics model parameterized by a real, hand-digitized FIS DD table -- and evaluates whether the model:

1. reconstructs known trick trajectories from their FIS code;
2. **generalizes** to trick codes withheld entirely from training;
3. produces **physically plausible** rotation trajectories consistent with the flip/twist count required by the FIS code;
4. supports **latent-space interpolation** between two real tricks to synthesize a novel, plausible "hybrid" maneuver;
5. outperforms a naive nearest-neighbor baseline and a small autoregressive Transformer baseline on held-out generalization.

The central research question is:

> **To what extent can a generative model, trained on biomechanically-grounded synthetic trick kinematics conditioned on real FIS difficulty codes, learn a latent representation that supports reconstruction of known aerial tricks and generation of novel, physically plausible ones?**

---
## 1. Introduction

### 1.1 Motivation

Freestyle aerial skiing (moguls aerials, big air) is judged in part using a codified system: every recognized maneuver is built from a base jump group (number of somersaults), modified by a number of twists, and assigned an official **Degree of Difficulty (DD)** value by the FIS. This compositional structure -- a small set of discrete building blocks (flip count, twist count) combined into a large space of named tricks, each with a real, published difficulty score -- makes aerial skiing well suited to **conditional generative modeling**: the conditioning signal is low-dimensional and interpretable, while the resulting kinematic trajectory is continuous and physically constrained by rigid-body rotation dynamics.

Real athlete motion-capture data for this sport is scarce: it requires specialized equipment, consenting elite athletes, and is rarely released publicly. A generative model that learns the mapping from trick *code* to trick *kinematics* well enough to generalize to codes it has never seen would be useful for coaching visualization, animation/game content generation, and judging-support tooling, none of which currently have access to a broad, systematic trick-trajectory library.

### 1.2 Research Question

> **Can a generative model, trained on biomechanically-grounded synthetic trick kinematics conditioned on real FIS difficulty codes, reconstruct known aerial tricks and generate novel, physically plausible ones for unseen codes?**

More specifically:

1. Can the model reconstruct a trick's rotational trajectory from its FIS code alone?
2. Does the model generalize to trick codes withheld entirely from training?
3. Do generated trajectories satisfy the physical rotation-count constraint implied by their FIS code (e.g. a "Full-Full" must complete two somersaults and two twists)?
4. Does interpolating between two real tricks in latent space produce a smooth, physically coherent "hybrid" maneuver?
5. How does a Conditional VAE compare against a naive nearest-neighbor baseline and a small autoregressive Transformer over quantized poses?

### 1.3 Objectives

1. Digitize a real FIS aerials Degree-of-Difficulty table.
2. Generate synthetic kinematic trajectories for each trick code using a rigid-body flight-dynamics model grounded in cited sports-biomechanics literature.
3. Train a Conditional VAE on this dataset, with entire trick classes held out to test generalization.
4. Implement and compare two baselines: a naive nearest-neighbor/noise baseline and a small autoregressive Transformer over quantized pose tokens.
5. Evaluate reconstruction quality, generalization, and physical plausibility.
6. Visualize the learned latent space and demonstrate latent interpolation between real tricks.
7. Discuss limitations and the gap between this synthetic-data approach and real motion capture.

### 1.4 Assumptions and Constraints

- **Synthetic-but-grounded kinematics:** no public motion-capture dataset exists for freestyle aerial skiing, so trajectories are generated from a simplified rigid-body flight model rather than measured directly. The model is grounded in real physical principles (conservation of angular momentum, tuck-driven moment-of-inertia change) documented in the cited literature.
- **Reduced-order representation:** each trick is represented by three time series (somersault angle, twist angle, tuck factor) rather than a full-body skeleton; this is a deliberate simplification for a course-scale project.
- **Real conditioning data:** flip count, twist count, and DD score are digitized directly from a real, published FIS-derived table (see Section 3).
- **Held-out generalization design:** two entire trick classes are excluded from training and used only for testing, to evaluate genuine generalization rather than memorization.

---
## 2. Literature Review

### 2.1 The FIS Aerial Trick Coding System

Every recognized aerial maneuver is identified by a specific code describing its basic jump group and modifiers (twists, position); the difficulty of the maneuver is established using a published Degree of Difficulty table, and jump codes are built by adding individual component values together. Source: FIS, *Freestyle Skiing Judging Handbook*, Edition November 2025, https://assets.fis-ski.com/f/252177/x/40158d23e4/freestyle-skiing-judging-handbook.pdf

A secondary, more compact digest of representative DD values across common maneuvers (used to construct the training-conditioning table in Section 3) is published by JudgeMate. Source: JudgeMate, *Aerials Difficulty Table & Scoring Guide*, https://www.judgemate.com/en/guides/aerials-degree-of-difficulty-table

For general competition-format context: the aerials scoring formula multiplies judged component scores by the maneuver's Degree of Difficulty to produce the final score. Source: NBC Olympics, *Freestyle Skiing 101: Rules, scoring and competition format*, https://www.nbcolympics.com/news/freestyle-skiing-101-what-know-about-olympic-aerials

### 2.2 Rigid-Body Mechanics of Aerial Rotation

The physics generator in Section 3 is grounded in the classical rigid-body treatment of somersaulting and twisting motion in aerial sports. A mathematical framework modeling an athlete as a system of coupled rigid bodies, using Euler's equations of motion generalized to non-rigid bodies, has been used to simulate and even design new twisting-somersault maneuvers in diving. Source: Tong, W., & Dullin, H. R. (2017). *A New Twisting Somersault: 513XD*. Journal of Nonlinear Science, 27(6), 2037-2061. https://doi.org/10.1007/s00332-017-9403-4

The specific mechanism used here for introducing twist mid-flight -- an asymmetric shape change producing "tilt", which is held to accumulate twist before being reversed to stop it -- follows the classical biomechanical partitioning of twisting somersaults developed for diving, gymnastics, and trampoline. Source: Yeadon, M. R. (1993). *The biomechanics of twisting somersaults. Part IV: Partitioning performances using the tilt angle*. Journal of Sports Sciences, 11(3), 219-225. https://doi.org/10.1080/02640419308729988

Related rigid-body flight biomechanics has also been studied extensively in ski jumping, where forward somersaulting angular momentum during take-off and early flight is a recognized topic of ongoing research. This supports the general modeling choice of treating in-flight rotation as governed by conserved angular momentum. Source: Virmavirta, M. (2016). *Biomechanics research in ski jumping*, summarized in *Science and Skiing*, https://www.tandfonline.com/doi/full/10.1080/14763140701687560

### 2.3 Generative Modeling Methods

The primary model in this project is a Conditional Variational Autoencoder, built on the variational autoencoder framework. Source: Kingma, D. P., & Welling, M. (2013). *Auto-Encoding Variational Bayes*. arXiv:1312.6114, https://arxiv.org/abs/1312.6114

The conditional extension used to condition generation on trick codes follows the structured-output conditional generative modeling framework. Source: Sohn, K., Lee, H., & Yan, X. (2015). *Learning Structured Output Representation using Deep Conditional Generative Models*. Advances in Neural Information Processing Systems 28 (NeurIPS 2015), 3483-3491.

The Transformer baseline (Section 6) follows the self-attention sequence architecture introduced for machine translation and since adopted broadly across sequence modeling. Source: Vaswani, A. et al. (2017). *Attention Is All You Need*. Advances in Neural Information Processing Systems 30 (NeurIPS 2017). arXiv:1706.03762, https://arxiv.org/abs/1706.03762

Directly relevant prior work in the specific application area -- learning generative kinematic models of human motion using autoregressive conditional VAEs -- has been demonstrated for character animation control, where a Motion VAE defines the action space governing a character's movement evolution over time. This project follows a similar conditional-VAE-over-kinematics approach at a much smaller scale. Source: Ling, H. Y., Zinno, F., Cheng, G., & van de Panne, M. (2020). *Character Controllers Using Motion VAEs*. ACM Transactions on Graphics (Proc. SIGGRAPH 2020), 39(4), Article 40. arXiv:2103.14274, https://arxiv.org/abs/2103.14274

Useful method/library references:

- PyTorch `nn.TransformerEncoderLayer`: https://docs.pytorch.org/docs/stable/generated/torch.nn.TransformerEncoderLayer.html
- scikit-learn `KMeans`: https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html
- scikit-learn `PCA`: https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html

---
## 3. Data Collection and Generation

### 3.1 Real-World Data: FIS Degree-of-Difficulty Table

The table below is hand-digitized from a published aerials Degree-of-Difficulty reference (JudgeMate's digest of the FIS-defined jump-coding system; see Section 2.1). Each row is a real, named, competition-recognized maneuver.

| Column | Meaning |
|---|---|
| `trick_name` | Official/shorthand maneuver name and jump code |
| `n_flips` | Number of somersaults |
| `n_twists` | Total number of twists across all somersaults |
| `dd_score` | Official FIS Degree of Difficulty value |

This is the only source of **real-world structured data** in the project, and it is used both to parameterize the synthetic kinematic generator (Section 3.2) and as the conditioning signal for the generative model (Section 5).

In [2]:
# ── 3.1 FIS DEGREE-OF-DIFFICULTY TABLE (real, hand-digitized reference data) ──
# Source: JudgeMate, "Aerials Difficulty Table & Scoring Guide",
# https://www.judgemate.com/en/guides/aerials-degree-of-difficulty-table
# (representative DD values; primary source is the FIS Freestyle Skiing
# Judging Handbook / International Competition Rules)

fis_dd_table = pd.DataFrame([
    {"trick_name": "Layout (bL)",                     "n_flips": 1, "n_twists": 0, "dd_score": 2.050},
    {"trick_name": "Full Twist (bF)",                  "n_flips": 1, "n_twists": 1, "dd_score": 2.900},
    {"trick_name": "Double Full Twist (bFF)",          "n_flips": 1, "n_twists": 2, "dd_score": 3.525},
    {"trick_name": "Back Double (bLbL)",               "n_flips": 2, "n_twists": 0, "dd_score": 3.150},
    {"trick_name": "Full-Full (bFbF)",                 "n_flips": 2, "n_twists": 2, "dd_score": 3.825},
    {"trick_name": "Full-Double Full (bFbFF)",         "n_flips": 2, "n_twists": 3, "dd_score": 4.175},
    {"trick_name": "Double Full-Full-Full (bFFbFbF)",  "n_flips": 3, "n_twists": 4, "dd_score": 4.653},
    {"trick_name": "Full-Full-Double Full (bFbFbFF)",  "n_flips": 3, "n_twists": 4, "dd_score": 4.653},
    {"trick_name": "Triple Full-Full-Full (bFFFbFbF)", "n_flips": 3, "n_twists": 5, "dd_score": 4.900},
    {"trick_name": "Quad Twisting Triple (bFFFFbFbF)", "n_flips": 3, "n_twists": 6, "dd_score": 5.000},
], columns=["trick_name", "n_flips", "n_twists", "dd_score"])
fis_dd_table["trick_id"] = np.arange(len(fis_dd_table))

# Persist the digitized reference table as a real file on disk, rather than
# only living as an in-memory literal. Downstream cells load it back from
# this file (Section 3.4), the same way a real data pipeline would.
os.makedirs("data", exist_ok=True)
fis_dd_table.to_csv("../data/fis_dd_table.csv", index=False)
print(f"Wrote {len(fis_dd_table)} rows to data/fis_dd_table.csv")

fis_dd_table

Wrote 10 rows to data/fis_dd_table.csv


,trick_name,n_flips,n_twists,dd_score,trick_id
0,Layout (bL),1,0,2.050,0
1,Full Twist (bF),1,1,2.900,1
2,Double Full Twist (bFF),1,2,3.525,2
3,Back Double (bLbL),2,0,3.150,3
4,Full-Full (bFbF),2,2,3.825,4
5,Full-Double Full (bFbFF),2,3,4.175,5
6,Double Full-Full-Full (bFFbFbF),3,4,4.653,6
7,Full-Full-Double Full (bFbFbFF),3,4,4.653,7
8,Triple Full-Full-Full (bFFFbFbF),3,5,4.900,8
9,Quad Twisting Triple (bFFFFbFbF),3,6,5.000,9


### 3.2 Physics-Based Synthetic Kinematic Trajectory Generator

Because no public motion-capture dataset exists for freestyle aerial skiing, each row of the FIS table above is expanded into a full kinematic trajectory using a simplified rigid-body flight model. Every trick is represented as a normalized-time sequence of three channels:

- **Somersault angle** `θ_som(t)` -- cumulative forward/backward rotation, in degrees
- **Twist angle** `θ_twist(t)` -- cumulative twist rotation, in degrees
- **Tuck factor** `τ(t) ∈ [0, 1]` -- 0 = fully extended (layout), 1 = fully tucked

**Flight time.** Longer, harder combinations require more air time to complete their required rotation:

`t_flight = 0.55 + 0.16 · n_flips + 0.03 · n_twists` (arbitrary normalized units, with athlete-to-athlete noise added)

**Somersault angular velocity via conserved angular momentum.** Once airborne, angular momentum about the somersault axis is conserved: `L = I(t) · ω(t) = constant`. Tucking lowers the effective moment of inertia `I(t)`, which increases the angular velocity `ω(t)` -- the same principle used to model twisting-somersault dynamics in diving (Tong & Dullin, 2017). The tuck-factor profile is a bell-shaped curve (extended at takeoff/landing, tucked mid-flight), and the somersault angle is obtained by integrating `ω(t) ∝ 1 / I(t)` over normalized time, scaled so the final angle equals `n_flips × 360°`.

**Twist angle via asymmetric tilt.** Twist is introduced within a bounded window inside the flight phase by an asymmetric shape change (e.g. one arm dropping), following the classical partitioning of twisting-somersault mechanics (Yeadon, 1993). The twist angle ramps linearly within this window up to `n_twists × 360°` and is then held constant.

**Inter-athlete variability.** Each generated instance adds independent Gaussian noise to the flight time, tuck depth, twist-window timing, and per-frame angles, so that multiple synthetic "performances" of the same trick code are all slightly different -- emulating real inter-athlete and inter-attempt variability.

In [3]:
# ── 3.2 PHYSICS-BASED SYNTHETIC KINEMATIC TRAJECTORY GENERATOR ──────────────
T_STEPS = 60  # normalized time steps per trick


def generate_trick_sequence(n_flips, n_twists, dd_score, rng, noise_scale=1.0):
    """Generate one synthetic (T_STEPS, 3) kinematic trajectory for a trick.

    Channels: [somersault_angle_deg, twist_angle_deg, tuck_factor]

    Grounded in conservation of angular momentum during free flight
    (Tong & Dullin, 2017) and asymmetric-tilt twist introduction
    (Yeadon, 1993). See Section 3.2 for the full derivation.
    """
    base_flight_time = 0.55 + 0.16 * n_flips + 0.03 * n_twists
    flight_time = base_flight_time * (1 + rng.normal(0, 0.04 * noise_scale))

    t = np.linspace(0, 1, T_STEPS)

    # Tuck factor: bell-shaped, deeper tuck for harder tricks
    tuck_depth = 0.35 + 0.55 * min(n_flips, 3) / 3.0
    tuck_depth *= (1 + rng.normal(0, 0.05 * noise_scale))
    tuck_depth = np.clip(tuck_depth, 0.1, 1.0)
    tuck_center, tuck_width = 0.5, 0.32
    tuck_factor = tuck_depth * np.exp(-((t - tuck_center) ** 2) / (2 * tuck_width ** 2))

    # Somersault angle: angular velocity ∝ 1 / moment_of_inertia (L = I*omega)
    inertia = 1.0 - 0.55 * tuck_factor
    omega_som_raw = 1.0 / inertia
    omega_som_raw /= omega_som_raw.mean()
    total_som_rotation = n_flips * 360.0
    som_profile = np.cumsum(omega_som_raw)
    som_profile = som_profile / som_profile[-1] * total_som_rotation
    som_profile += rng.normal(0, 2.0 * noise_scale, size=T_STEPS)

    # Twist angle: ramps within an asymmetric-tilt window, then holds
    twist_angle = np.zeros(T_STEPS)
    if n_twists > 0:
        window_start = np.clip(0.30 + rng.normal(0, 0.02 * noise_scale), 0, 1)
        window_end = np.clip(0.80 + rng.normal(0, 0.02 * noise_scale), 0, 1)
        total_twist_rotation = n_twists * 360.0
        in_window = (t >= window_start) & (t <= window_end)
        ramp = np.zeros(T_STEPS)
        if in_window.sum() > 1:
            local_t = (t[in_window] - window_start) / max(window_end - window_start, 1e-6)
            ramp[in_window] = local_t
            ramp[t > window_end] = 1.0
        twist_angle = ramp * total_twist_rotation
        twist_angle += rng.normal(0, 2.0 * noise_scale, size=T_STEPS)

    sequence = np.stack([som_profile, twist_angle, tuck_factor], axis=1).astype(np.float32)
    return sequence, flight_time


# Sanity check: final rotation should match the FIS-required flip/twist count
seq, ft = generate_trick_sequence(2, 2, 3.825, rng)
print(f"Full-Full (2 flips, 2 twists) -> final somersault={seq[-1,0]:.1f} deg "
      f"(target 720), final twist={seq[-1,1]:.1f} deg (target 720), flight_time={ft:.2f}")

Full-Full (2 flips, 2 twists) -> final somersault=719.3 deg (target 720), final twist=722.7 deg (target 720), flight_time=0.94


### 3.3 Dataset Generation and Persistence

For each of the 10 real trick codes, 40 synthetic instances are generated (400 sequences total). Two entire trick classes -- **Double Full-Full-Full** and **Triple Full-Full-Full** -- are withheld completely from training. This is the key test of generalization: the model never sees a single training example of these codes, only their `[n_flips, n_twists, dd_score]` conditioning vector at generation time. The remaining 8 trick classes are split 80/20 into train and test sets.

Rather than keeping this dataset only as in-memory Python variables, it is written to `data/synthetic_trick_sequences.npz` -- a real file on disk -- including the train/test/held-out split indices and normalization statistics, so the split is fixed and reproducible rather than silently re-randomized if cells are re-run out of order.

In [8]:
# ── 3.3 GENERATE THE FULL SYNTHETIC DATASET AND SAVE IT TO DISK ─────────────
N_INSTANCES_PER_TRICK = 40
HELD_OUT_TRICK_IDS = [6, 8]  # "Double Full-Full-Full" and "Triple Full-Full-Full"

# Read the real trick table back from disk (data/fis_dd_table.csv) rather than
# reusing the in-memory dataframe, so the generator is driven by the on-disk
# file, exactly as it would be in a run starting fresh from Section 3.4.
_fis_table_for_generation = pd.read_csv("../data/fis_dd_table.csv")


def build_dataset(fis_dd_table, n_instances, rng):
    sequences, conditions, ids = [], [], []
    for _, row in fis_dd_table.iterrows():
        for _ in range(n_instances):
            seq, _ = generate_trick_sequence(row.n_flips, row.n_twists, row.dd_score, rng)
            sequences.append(seq)
            conditions.append([row.n_flips, row.n_twists, row.dd_score])
            ids.append(row.trick_id)
    return (np.stack(sequences), np.array(conditions, dtype=np.float32), np.array(ids))


_X, _C, _trick_ids = build_dataset(_fis_table_for_generation, N_INSTANCES_PER_TRICK, rng)

_x_mean, _x_std = _X.mean(axis=(0, 1)), _X.std(axis=(0, 1))
_c_mean, _c_std = _C.mean(axis=0), _C.std(axis=0)

_seen_mask = ~np.isin(_trick_ids, HELD_OUT_TRICK_IDS)
_held_mask = np.isin(_trick_ids, HELD_OUT_TRICK_IDS)
_idx_seen = np.where(_seen_mask)[0]
rng.shuffle(_idx_seen)
_n_train = int(0.8 * len(_idx_seen))
_train_idx, _test_idx = _idx_seen[:_n_train], _idx_seen[_n_train:]
_held_idx = np.where(_held_mask)[0]

np.savez(
    "../data/synthetic_trick_sequences.npz",
    X=_X, C=_C, trick_ids=_trick_ids,
    x_mean=_x_mean, x_std=_x_std, c_mean=_c_mean, c_std=_c_std,
    train_idx=_train_idx, test_idx=_test_idx, held_idx=_held_idx,
    held_out_trick_ids=np.array(HELD_OUT_TRICK_IDS),
)
print(f"Wrote {_X.shape[0]} sequences to ../data/synthetic_trick_sequences.npz")
print(f"  train={len(_train_idx)}  test={len(_test_idx)}  held-out (unseen tricks)={len(_held_idx)}")

Wrote 400 sequences to ../data/synthetic_trick_sequences.npz
  train=256  test=64  held-out (unseen tricks)=80


### 3.4 Loading the Dataset From Disk

From this point on, the notebook works exactly like a typical ML pipeline: it reads the real trick table and the generated kinematic dataset back from the files written in Section 3.3, rather than referencing the in-memory objects created above. If you restart the kernel and run only from this cell onward (with `data/` already populated), everything below still works.

In [9]:
# ── 3.4 LOAD THE FIS TABLE AND SYNTHETIC DATASET FROM DISK ──────────────────
fis_dd_table = pd.read_csv("../data/fis_dd_table.csv")

_data = np.load("../data/synthetic_trick_sequences.npz")
X, C, trick_ids = _data["X"], _data["C"], _data["trick_ids"]
x_mean, x_std = _data["x_mean"], _data["x_std"]
c_mean, c_std = _data["c_mean"], _data["c_std"]
train_idx, test_idx, held_idx = _data["train_idx"], _data["test_idx"], _data["held_idx"]
HELD_OUT_TRICK_IDS = _data["held_out_trick_ids"].tolist()

print("Loaded from disk -> X:", X.shape, " C:", C.shape, " trick_ids:", trick_ids.shape)
print(f"train={len(train_idx)}  test={len(test_idx)}  held-out (unseen tricks)={len(held_idx)}")
print("Held-out trick classes:", fis_dd_table[fis_dd_table.trick_id.isin(HELD_OUT_TRICK_IDS)].trick_name.tolist())

seen_mask = ~np.isin(trick_ids, HELD_OUT_TRICK_IDS)

Xn = (X - x_mean) / x_std
Cn = (C - c_mean) / c_std

X_t = torch.tensor(Xn.reshape(len(Xn), -1), dtype=torch.float32)
C_t = torch.tensor(Cn, dtype=torch.float32)
D_IN, D_COND = X_t.shape[1], C_t.shape[1]

Loaded from disk -> X: (400, 60, 3)  C: (400, 3)  trick_ids: (400,)
train=256  test=64  held-out (unseen tricks)=80
Held-out trick classes: ['Double Full-Full-Full (bFFbFbF)', 'Triple Full-Full-Full (bFFFbFbF)']
